# Optimización de Campañas Publicitarias
## Sistema analítico: predicción de conversión, ROI, recomendación de inversión, A/B testing y atribución

**Dataset:** Facebook Ads Campaign (Kaggle — `madislemsalu/facebook-ad-campaign`)

**Objetivo general:** desarrollar un sistema analítico que optimice la inversión publicitaria mediante modelos
predictivos y análisis de atribución, maximizando el retorno de inversión.

**Integrantes:** Evelin Dayanna Pulsara Madroñero · Deivid Julián Alvarado Morán · Giovanny Andrés Zambrano ·
Esteban Giovanny Meneses Paz

| Sección | Contenido | Responsable |
|---|---|---|
| 1 | Carga y limpieza de datos | Evelin |
| 2 | KPIs de campaña (CTR, CPC, CPM, CPA, ROI) | Evelin |
| 3 | Análisis exploratorio | Evelin |
| 4 | Preparación para modelado | Evelin |
| 5 | Random Forest y AdaBoost (clasificación) con búsqueda de hiperparámetros | Evelin |
| 6 | Evaluación y comparación | Evelin |
| 7 | Importancia de variables | Evelin |
| 8 | Modelos de regresión (conversiones esperadas) | Evelin |
| 9 | Análisis de ROI del modelo | Evelin |
| 10 | Recomendación de inversión por segmento | Evelin |
| 11 | A/B testing | Deivid |
| 12 | Atribución y comparación de canales | Giovanny Andrés |
| 13 | Exportación de resultados para la app web | Evelin |
| 14 | Conclusiones finales | Equipo |

La app web (dashboard de campañas) usa los archivos que exporta la sección 13.


## 0. Librerías

In [ ]:
import json, os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              RandomForestRegressor, AdaBoostRegressor)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, mean_absolute_error, r2_score)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
np.random.seed(SEED)

## 1. Carga y limpieza de datos

En Colab se descarga directo de Kaggle con `kagglehub`. Si falla, sube el CSV manualmente
(menú de archivos a la izquierda) y la celda lo encuentra.

In [ ]:
def localizar_csv():
    locales = [p for p in glob.glob("*.csv") + glob.glob("data/*.csv")
               if not p.startswith(("datos_limpios", "kpis_", "predicciones_"))]
    if locales:
        return locales[0]
    try:
        import kagglehub
        carpeta = kagglehub.dataset_download("madislemsalu/facebook-ad-campaign")
        return glob.glob(os.path.join(carpeta, "**", "*.csv"), recursive=True)[0]
    except Exception as e:
        raise FileNotFoundError("No se encontró el CSV. Súbelo a Colab y vuelve a ejecutar.") from e

RUTA = localizar_csv()
df_raw = pd.read_csv(RUTA)
print("Archivo:", RUTA, "| Forma:", df_raw.shape)
df_raw.head()

**Normalización de columnas.** Existen dos versiones públicas de este dataset con nombres distintos
(`xyz_campaign_id`/`interest` vs `campaign_id`/`interest1..3`). Se unifican para que el resto del
código funcione con cualquiera.

In [ ]:
df = df_raw.copy()
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(" ", "_").str.replace("-", "_"))
df = df.rename(columns={"xyz_campaign_id": "campaign_id", "interest": "interest1"})

NUMERICAS = ["impressions", "clicks", "spent", "total_conversion", "approved_conversion"]
for c in NUMERICAS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df.dtypes)
print("\nNulos por columna:\n", df.isna().sum())

**Filas corruptas.** En la versión de Kaggle algunas filas vienen desplazadas (faltan columnas y los
valores quedan corridos). Esas filas no son confiables: se identifican y se eliminan, dejando registro
de cuántas fueron.

In [ ]:
n_inicial = len(df)
mascara_valida = df[["campaign_id", "age", "gender"] + NUMERICAS].notna().all(axis=1)
mascara_valida &= df["gender"].astype(str).str.upper().isin(["M", "F"])
mascara_valida &= df["age"].astype(str).str.match(r"^\d{2}-\d{2}$")
df = df[mascara_valida].copy()

df = df[(df["impressions"] > 0) & (df["spent"] >= 0) & (df["clicks"] >= 0)]
df = df.drop_duplicates(subset="ad_id") if "ad_id" in df.columns else df.drop_duplicates()

df["campaign_id"] = df["campaign_id"].astype(int).astype(str)
df["gender"] = df["gender"].str.upper()
for c in ["interest1", "interest2", "interest3"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(-1).astype(int).astype(str)

print(f"Filas iniciales: {n_inicial} | eliminadas: {n_inicial - len(df)} | finales: {len(df)}")
df.describe(include="all").T

## 2. KPIs de campaña

| KPI | Fórmula |
|---|---|
| CTR | clics / impresiones |
| CPC | gasto / clics |
| CPM | gasto / impresiones × 1000 |
| Tasa de conversión | conversiones aprobadas / clics |
| CPA | gasto / conversiones aprobadas |
| ROI | (ingreso − gasto) / gasto |

El dataset **no trae ingresos**, así que el ROI necesita un supuesto: el valor de una conversión
aprobada (`VALOR_CONVERSION`). Cámbialo y todo el notebook se recalcula. Documentar este supuesto en el informe.

In [ ]:
VALOR_CONVERSION = 100.0

def dividir(a, b):
    return np.where(b > 0, a / np.where(b > 0, b, 1), np.nan)

def agregar_kpis(d):
    d = d.copy()
    d["ctr"] = dividir(d["clicks"], d["impressions"])
    d["cpc"] = dividir(d["spent"], d["clicks"])
    d["cpm"] = dividir(d["spent"], d["impressions"]) * 1000
    d["conv_rate"] = dividir(d["approved_conversion"], d["clicks"])
    d["cpa"] = dividir(d["spent"], d["approved_conversion"])
    d["ingreso"] = d["approved_conversion"] * VALOR_CONVERSION
    d["roi"] = dividir(d["ingreso"] - d["spent"], d["spent"])
    return d

df = agregar_kpis(df)

def kpis_por(grupo):
    g = df.groupby(grupo)[["impressions", "clicks", "spent", "approved_conversion"]].sum()
    g = agregar_kpis(g)
    g["anuncios"] = df.groupby(grupo).size()
    return g.round(4)

kpis_campana = kpis_por("campaign_id")
kpis_edad = kpis_por("age")
kpis_genero = kpis_por("gender")
kpis_campana

## 3. Análisis exploratorio

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

kpis_campana["roi"].plot.bar(ax=ax[0, 0], color="#5B6CFF")
ax[0, 0].axhline(0, color="black", lw=1)
ax[0, 0].set_title(f"ROI por campaña (valor conversión = {VALOR_CONVERSION})")

kpis_edad["cpa"].plot.bar(ax=ax[0, 1], color="#FF7A59")
ax[0, 1].set_title("CPA por rango de edad")

sns.scatterplot(data=df, x="spent", y="approved_conversion", hue="campaign_id", alpha=.6, ax=ax[1, 0])
ax[1, 0].set_title("Gasto vs conversiones aprobadas")

sns.heatmap(df[["impressions", "clicks", "spent", "total_conversion",
                "approved_conversion", "ctr", "cpc"]].corr(),
            annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[1, 1])
ax[1, 1].set_title("Correlaciones")
plt.tight_layout(); plt.show()

**Preguntas para redactar hallazgos:** ¿qué campaña tiene mejor ROI? ¿qué grupo de edad convierte más
barato? ¿hay rendimientos decrecientes al aumentar el gasto?

## 4. Preparación para modelado

**Variable objetivo (clasificación):** `convierte = 1` si el anuncio tuvo al menos una conversión aprobada.

**Fuga de información:** `total_conversion` se excluye porque sólo se conoce después del resultado y está
casi perfectamente correlacionada con el objetivo. También se excluyen `conv_rate`, `cpa`, `ingreso` y `roi`
porque se calculan con el objetivo.

In [ ]:
df["convierte"] = (df["approved_conversion"] >= 1).astype(int)
print(df["convierte"].value_counts(normalize=True).rename("proporción"))

CATEGORICAS = [c for c in ["campaign_id", "age", "gender", "interest1", "interest2", "interest3"] if c in df.columns]
NUM_FEATURES = ["impressions", "clicks", "spent", "ctr", "cpc", "cpm"]

df[["cpc"]] = df[["cpc"]].fillna(0)
X = df[CATEGORICAS + NUM_FEATURES]
y = df["convierte"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
print("Train:", X_train.shape, "| Test:", X_test.shape)

preprocesador = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), CATEGORICAS),
    ("num", "passthrough", NUM_FEATURES),
])

## 5. Modelos de clasificación

Se usa `GridSearchCV` con validación cruzada estratificada de 5 particiones, optimizando **ROC-AUC**
(más robusto que accuracy si las clases no están balanceadas).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

rf_pipe = Pipeline([("prep", preprocesador),
                    ("model", RandomForestClassifier(random_state=SEED, n_jobs=-1))])
rf_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 6, 10],
    "model__min_samples_leaf": [1, 3, 5],
    "model__class_weight": [None, "balanced"],
}
rf_search = GridSearchCV(rf_pipe, rf_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
rf_search.fit(X_train, y_train)
print("Random Forest — mejores parámetros:", rf_search.best_params_)
print("AUC CV:", round(rf_search.best_score_, 4))

In [ ]:
ada_pipe = Pipeline([("prep", preprocesador),
                     ("model", AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=SEED),
                                                  random_state=SEED))])
ada_grid = {
    "model__n_estimators": [100, 300],
    "model__learning_rate": [0.05, 0.1, 0.5, 1.0],
    "model__estimator__max_depth": [1, 2, 3],
}
ada_search = GridSearchCV(ada_pipe, ada_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
ada_search.fit(X_train, y_train)
print("AdaBoost — mejores parámetros:", ada_search.best_params_)
print("AUC CV:", round(ada_search.best_score_, 4))

## 6. Evaluación y comparación en el conjunto de prueba

In [ ]:
modelos = {"Random Forest": rf_search.best_estimator_, "AdaBoost": ada_search.best_estimator_}

def evaluar(nombre, modelo):
    proba = modelo.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {"modelo": nombre,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba)}

comparacion = pd.DataFrame([evaluar(n, m) for n, m in modelos.items()]).set_index("modelo").round(4)
comparacion

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
for i, (nombre, modelo) in enumerate(modelos.items()):
    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test, ax=ax[i],
                                          display_labels=["No convierte", "Convierte"], cmap="Blues")
    ax[i].set_title(f"Matriz de confusión — {nombre}")
    fpr, tpr, _ = roc_curve(y_test, modelo.predict_proba(X_test)[:, 1])
    ax[2].plot(fpr, tpr, label=f"{nombre} (AUC={comparacion.loc[nombre, 'roc_auc']:.3f})")
ax[2].plot([0, 1], [0, 1], "k--")
ax[2].set_title("Curvas ROC"); ax[2].set_xlabel("FPR"); ax[2].set_ylabel("TPR"); ax[2].legend()
plt.tight_layout(); plt.show()

for nombre, modelo in modelos.items():
    print(f"=== {nombre} ===")
    print(classification_report(y_test, modelo.predict(X_test),
                                target_names=["No convierte", "Convierte"]))

In [ ]:
mejor_nombre = comparacion["roc_auc"].idxmax()
mejor_modelo = modelos[mejor_nombre]
print("Modelo seleccionado:", mejor_nombre)

## 7. Importancia de variables

Se usa **importancia por permutación** sobre el conjunto de prueba: mide cuánto cae el AUC si se
desordena cada variable. Es comparable entre ambos modelos (a diferencia de `feature_importances_`).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
importancias = {}
for i, (nombre, modelo) in enumerate(modelos.items()):
    r = permutation_importance(modelo, X_test, y_test, scoring="roc_auc",
                               n_repeats=15, random_state=SEED, n_jobs=-1)
    imp = pd.Series(r.importances_mean, index=X_test.columns).sort_values()
    importancias[nombre] = imp.round(4).to_dict()
    imp.plot.barh(ax=ax[i], xerr=r.importances_std[imp.index.map(list(X_test.columns).index)],
                  color="#5B6CFF")
    ax[i].set_title(f"Importancia por permutación — {nombre}")
plt.tight_layout(); plt.show()

## 8. Regresión: ¿cuántas conversiones aprobadas se esperan?

La clasificación dice *si* un anuncio convertirá; la regresión estima *cuántas* conversiones, lo que
permite estimar ingreso y ROI esperado. Esto lo usa la recomendación de inversión del dashboard.

In [ ]:
y_reg = df["approved_conversion"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y_reg, test_size=0.25,
                                                        stratify=y, random_state=SEED)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)

regresores = {
    "RF Regressor": GridSearchCV(
        Pipeline([("prep", preprocesador), ("model", RandomForestRegressor(random_state=SEED, n_jobs=-1))]),
        {"model__n_estimators": [300], "model__max_depth": [None, 8], "model__min_samples_leaf": [1, 3, 5]},
        cv=cv_reg, scoring="neg_mean_absolute_error", n_jobs=-1),
    "AdaBoost Regressor": GridSearchCV(
        Pipeline([("prep", preprocesador),
                  ("model", AdaBoostRegressor(estimator=DecisionTreeRegressor(random_state=SEED), random_state=SEED))]),
        {"model__n_estimators": [100, 300], "model__learning_rate": [0.05, 0.1, 0.5],
         "model__estimator__max_depth": [2, 3, 4]},
        cv=cv_reg, scoring="neg_mean_absolute_error", n_jobs=-1),
}

filas = []
for nombre, busqueda in regresores.items():
    busqueda.fit(Xr_train, yr_train)
    pred = np.clip(busqueda.predict(Xr_test), 0, None)
    filas.append({"modelo": nombre, "MAE": mean_absolute_error(yr_test, pred),
                  "R2": r2_score(yr_test, pred), "mejores_params": busqueda.best_params_})
comparacion_reg = pd.DataFrame(filas).set_index("modelo")
comparacion_reg[["MAE", "R2"]].round(4)

In [ ]:
mejor_reg_nombre = comparacion_reg["MAE"].astype(float).idxmin()
mejor_regresor = regresores[mejor_reg_nombre].best_estimator_
print("Regresor seleccionado:", mejor_reg_nombre)

## 9. Análisis de ROI del modelo

Pregunta de negocio: **si sólo hubiéramos invertido en los anuncios que el modelo predice que convierten,
¿cuánto habría mejorado el ROI frente a invertir en todos?**

Se ordenan los anuncios de prueba (que el modelo no vio) por probabilidad de conversión y se calcula el ROI
si sólo se invirtiera en el top 10 %, 20 %, … 100 %.

In [ ]:
test = df.loc[X_test.index].copy()
test["proba"] = mejor_modelo.predict_proba(X_test)[:, 1]

def roi_cartera(d):
    gasto, ingreso = d["spent"].sum(), d["ingreso"].sum()
    return pd.Series({"anuncios": len(d), "gasto": gasto, "ingreso": ingreso,
                      "conversiones": d["approved_conversion"].sum(),
                      "roi": (ingreso - gasto) / gasto if gasto > 0 else np.nan,
                      "cpa": gasto / d["approved_conversion"].sum() if d["approved_conversion"].sum() > 0 else np.nan})

base = roi_cartera(test)
ranking = test.sort_values("proba", ascending=False)
porcentajes = list(range(10, 101, 10))
simulacion = pd.DataFrame({p: roi_cartera(ranking.head(max(1, int(len(ranking) * p / 100))))
                           for p in porcentajes}).T
simulacion.index.name = "top_%_anuncios"
simulacion["umbral_proba"] = [ranking["proba"].iloc[max(1, int(len(ranking) * p / 100)) - 1] for p in porcentajes]
simulacion["%_gasto_usado"] = simulacion["gasto"] / base["gasto"] * 100
simulacion["%_conversiones_retenidas"] = simulacion["conversiones"] / base["conversiones"] * 100

print("ROI invirtiendo en todos los anuncios:", round(base["roi"], 4), "| CPA:", round(base["cpa"], 2))
simulacion.round(3)

In [ ]:
mejor_top = simulacion[simulacion["anuncios"] >= 10]["roi"].idxmax()
fila = simulacion.loc[mejor_top]
umbral_optimo = round(float(fila["umbral_proba"]), 4)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(simulacion.index, simulacion["roi"], marker="o", color="#5B6CFF", label="ROI con modelo")
ax1.axhline(base["roi"], color="gray", ls="--", label="ROI sin modelo")
ax1.axvline(mejor_top, color="green", ls=":", label=f"Mejor: top {mejor_top}%")
ax1.set_xlabel("Top % de anuncios elegidos por el modelo"); ax1.set_ylabel("ROI")
ax2 = ax1.twinx()
ax2.plot(simulacion.index, simulacion["%_conversiones_retenidas"], color="orange", ls="-.",
         label="% conversiones retenidas")
ax2.set_ylabel("% conversiones retenidas")
ax1.legend(loc="lower right"); ax2.legend(loc="upper right")
plt.title(f"ROI según % de anuncios seleccionados — {mejor_nombre}")
plt.tight_layout(); plt.show()

print(f"Invirtiendo sólo en el top {mejor_top}% (probabilidad ≥ {umbral_optimo}): ROI {fila['roi']:.3f} "
      f"vs {base['roi']:.3f} sin modelo, usando {fila['%_gasto_usado']:.1f}% del gasto y reteniendo "
      f"{fila['%_conversiones_retenidas']:.1f}% de las conversiones.")

**Lectura:** elegir un top más pequeño concentra el presupuesto en anuncios con más probabilidad de convertir (mejor ROI),
pero se pierden conversiones. El punto óptimo es un compromiso que se debe justificar en el informe.

**Limitación:** es una simulación retrospectiva; asume que el rendimiento de cada anuncio no cambia al
reasignar presupuesto.

## 10. Recomendación de inversión

**Pregunta:** dado un presupuesto, ¿cuánto invertir en cada segmento (campaña × edad × género) para
maximizar las conversiones y el ROI?

**Método:**
1. Para cada segmento se arma un *anuncio típico* con su CPM y CTR históricos y se le pide al
   **modelo de regresión** cuántas conversiones daría con distintos niveles de gasto. Eso forma una
   **curva de respuesta** gasto → conversiones.
2. Los árboles dan curvas en escalones, así que se usa su **envolvente cóncava**: una curva suave con
   rendimientos decrecientes (cada peso extra rinde igual o menos que el anterior).
3. El presupuesto se reparte de forma **voraz**: cada peso va al segmento donde rinde más conversiones.
   Con curvas cóncavas este reparto es óptimo (es una mochila fraccionaria).
4. Si `solo_rentable=True`, se deja de invertir cuando el siguiente peso ya no se recupera
   (conversiones marginales × valor de conversión ≤ 1).

Se compara contra dos repartos ingenuos: **igual** entre segmentos y **proporcional al gasto histórico**.

El código queda en `salidas/recomendador.py` para que el dashboard lo importe directamente.

In [ ]:
os.makedirs("salidas", exist_ok=True)

In [ ]:
%%writefile salidas/recomendador.py
import numpy as np
import pandas as pd
import joblib


class RecomendadorInversion:
    """Reparte un presupuesto entre segmentos (campaña x edad x género)
    maximizando las conversiones esperadas según el modelo de regresión."""

    def __init__(self, modelo, datos, valor_conversion,
                 segmento=("campaign_id", "age", "gender"),
                 puntos=40, factor_max=1.5, min_anuncios=3):
        self.modelo = modelo
        self.datos = datos.copy()
        self.valor = float(valor_conversion)
        self.segmento = list(segmento)
        self.puntos = puntos
        self.factor_max = factor_max
        self.min_anuncios = min_anuncios
        self.columnas = list(modelo.feature_names_in_)
        for c in self.segmento:
            self.datos[c] = self.datos[c].astype(str)
        self.curvas = self._construir_curvas()

    @classmethod
    def desde_archivos(cls, ruta_modelo, ruta_datos, valor_conversion, **kwargs):
        datos = pd.read_csv(ruta_datos, dtype={"campaign_id": str, "interest1": str,
                                               "interest2": str, "interest3": str})
        return cls(joblib.load(ruta_modelo), datos, valor_conversion, **kwargs)

    @staticmethod
    def _envolvente_concava(xs, ys):
        hull = []
        for p in zip(xs, ys):
            while len(hull) >= 2:
                (ox, oy), (ax, ay) = hull[-2], hull[-1]
                if (ax - ox) * (p[1] - oy) - (ay - oy) * (p[0] - ox) >= 0:
                    hull.pop()
                else:
                    break
            hull.append(p)
        hx, hy = zip(*hull)
        return np.array(hx), np.array(hy)

    def _anuncios_simulados(self, grupo, gastos):
        imp, clk, gas = grupo["impressions"].sum(), grupo["clicks"].sum(), grupo["spent"].sum()
        cpm = gas / imp * 1000 if imp > 0 else 0
        ctr = clk / imp if imp > 0 else 0
        impresiones = np.where(cpm > 0, gastos / max(cpm, 1e-9) * 1000, 0)
        clics = impresiones * ctr
        numericas = {"impressions", "clicks", "spent", "ctr", "cpm", "cpc"}
        base = {c: str(grupo[c].mode().iloc[0]) for c in self.columnas if c not in numericas}
        filas = pd.DataFrame({**base,
                              "impressions": impresiones, "clicks": clics, "spent": gastos,
                              "ctr": ctr, "cpm": cpm,
                              "cpc": np.where(clics > 0, gastos / np.where(clics > 0, clics, 1), 0)})
        return filas[self.columnas]

    def _construir_curvas(self):
        curvas = {}
        for clave, grupo in self.datos.groupby(self.segmento):
            if len(grupo) < self.min_anuncios or grupo["spent"].max() <= 0:
                continue
            gastos = np.linspace(0, grupo["spent"].max() * self.factor_max, self.puntos)
            conv = np.clip(self.modelo.predict(self._anuncios_simulados(grupo, gastos)), 0, None)
            conv[0] = 0
            conv = np.maximum.accumulate(conv)
            hx, hy = self._envolvente_concava(gastos, conv)
            curvas[clave] = {"n_anuncios": len(grupo), "gastos": gastos, "conv": conv,
                             "hx": hx, "hy": hy,
                             "gasto_historico": grupo["spent"].sum(),
                             "conv_historicas": grupo["approved_conversion"].sum()}
        return curvas

    def conversiones_esperadas(self, clave, inversion):
        c = self.curvas[clave]
        n = c["n_anuncios"]
        return n * np.interp(inversion / n, c["hx"], c["hy"])

    def capacidad_maxima(self):
        return sum(c["n_anuncios"] * c["hx"][-1] for c in self.curvas.values())

    def recomendar(self, presupuesto, solo_rentable=True):
        tramos = []
        for clave, c in self.curvas.items():
            dx, dy = np.diff(c["hx"]), np.diff(c["hy"])
            for a, b in zip(dx, dy):
                if a > 0:
                    tramos.append((b / a, clave, a * c["n_anuncios"]))
        tramos.sort(key=lambda t: t[0], reverse=True)

        asignado, restante = {}, float(presupuesto)
        for pendiente, clave, capacidad in tramos:
            if restante <= 0 or (solo_rentable and pendiente * self.valor <= 1):
                break
            monto = min(capacidad, restante)
            asignado[clave] = asignado.get(clave, 0.0) + monto
            restante -= monto
        return self.evaluar(asignado, presupuesto)

    def reparto_historico(self, presupuesto):
        total = sum(c["gasto_historico"] for c in self.curvas.values())
        return self.evaluar({k: presupuesto * c["gasto_historico"] / total
                             for k, c in self.curvas.items()}, presupuesto)

    def reparto_igual(self, presupuesto):
        return self.evaluar({k: presupuesto / len(self.curvas) for k in self.curvas}, presupuesto)

    def evaluar(self, asignacion, presupuesto):
        filas = []
        for clave, inv in asignacion.items():
            if inv <= 0:
                continue
            conv = self.conversiones_esperadas(clave, inv)
            filas.append({**dict(zip(self.segmento, clave)),
                          "inversion": inv, "conversiones_esperadas": conv,
                          "ingreso_esperado": conv * self.valor})
        tabla = pd.DataFrame(filas)
        if tabla.empty:
            tabla = pd.DataFrame(columns=self.segmento + ["inversion", "conversiones_esperadas",
                                                          "ingreso_esperado"])
        tabla["roi_esperado"] = (tabla["ingreso_esperado"] - tabla["inversion"]) / tabla["inversion"]
        tabla["%_presupuesto"] = tabla["inversion"] / presupuesto * 100
        tabla = tabla.sort_values("inversion", ascending=False).reset_index(drop=True)
        invertido = tabla["inversion"].sum()
        ingreso = tabla["ingreso_esperado"].sum()
        resumen = {"presupuesto": float(presupuesto),
                   "invertido": float(invertido),
                   "sin_invertir": float(presupuesto - invertido),
                   "conversiones_esperadas": float(tabla["conversiones_esperadas"].sum()),
                   "ingreso_esperado": float(ingreso),
                   "roi_esperado": float((ingreso - invertido) / invertido) if invertido > 0 else 0.0}
        return tabla.round(3), resumen

In [ ]:
import sys, importlib
sys.path.insert(0, "salidas")
import recomendador
importlib.reload(recomendador)

reco = recomendador.RecomendadorInversion(mejor_regresor, df, VALOR_CONVERSION)
print("Segmentos con curva:", len(reco.curvas))
print("Capacidad máxima de inversión:", round(reco.capacidad_maxima(), 2))

In [ ]:
top_seg = sorted(reco.curvas, key=lambda k: reco.curvas[k]["conv_historicas"], reverse=True)[:6]
fig, ax = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
for a, clave in zip(ax.ravel(), top_seg):
    c = reco.curvas[clave]
    a.step(c["gastos"], c["conv"], where="post", color="lightgray", label="Modelo (escalones)")
    a.plot(c["hx"], c["hy"], color="#5B6CFF", marker="o", ms=3, label="Envolvente cóncava")
    a.set_title(" | ".join(clave)); a.set_xlabel("Gasto por anuncio"); a.set_ylabel("Conversiones")
ax[0, 0].legend()
plt.suptitle("Curvas de respuesta por segmento"); plt.tight_layout(); plt.show()

In [ ]:
PRESUPUESTO = round(df["spent"].sum() * 0.25, -2)

tabla_reco, resumen_reco = reco.recomendar(PRESUPUESTO, solo_rentable=True)
print(f"Presupuesto: {PRESUPUESTO:,.0f}")
for k, v in resumen_reco.items():
    print(f"  {k}: {v:,.3f}")
tabla_reco.head(15)

In [ ]:
_, r_hist = reco.reparto_historico(PRESUPUESTO)
_, r_igual = reco.reparto_igual(PRESUPUESTO)
_, r_todo = reco.recomendar(PRESUPUESTO, solo_rentable=False)

estrategias = pd.DataFrame({
    "Igual por segmento": r_igual,
    "Proporcional al histórico": r_hist,
    "Modelo (invertir todo)": r_todo,
    "Modelo (solo rentable)": resumen_reco,
}).T
display(estrategias[["invertido", "conversiones_esperadas", "ingreso_esperado", "roi_esperado"]].round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
estrategias["conversiones_esperadas"].plot.barh(ax=ax[0], color=["#B0B7C3", "#B0B7C3", "#5B6CFF", "#2BB673"])
ax[0].set_title("Conversiones esperadas")
estrategias["roi_esperado"].plot.barh(ax=ax[1], color=["#B0B7C3", "#B0B7C3", "#5B6CFF", "#2BB673"])
ax[1].axvline(0, color="black", lw=1); ax[1].set_title("ROI esperado")
plt.tight_layout(); plt.show()

In [ ]:
presupuestos = np.linspace(PRESUPUESTO * 0.2, min(PRESUPUESTO * 4, reco.capacidad_maxima()), 12)
curva_ppto = pd.DataFrame([{"presupuesto": p,
                            "roi_modelo": reco.recomendar(p, solo_rentable=False)[1]["roi_esperado"],
                            "roi_historico": reco.reparto_historico(p)[1]["roi_esperado"],
                            "invertido_rentable": reco.recomendar(p)[1]["invertido"]}
                           for p in presupuestos])

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(curva_ppto["presupuesto"], curva_ppto["roi_modelo"], marker="o", label="Reparto del modelo")
ax.plot(curva_ppto["presupuesto"], curva_ppto["roi_historico"], marker="s", label="Reparto histórico")
ax.axhline(0, color="black", lw=1)
ax.set_xlabel("Presupuesto"); ax.set_ylabel("ROI esperado"); ax.legend()
ax.set_title("ROI esperado según el tamaño del presupuesto")
plt.tight_layout(); plt.show()

punto_saturacion = curva_ppto["invertido_rentable"].max()
print(f"A partir de ~{punto_saturacion:,.0f} de inversión, el siguiente peso ya no es rentable.")

**Lectura para el informe:**
- La tabla dice **dónde** poner cada peso; el gráfico de estrategias muestra **cuánto mejora** frente a repartir a ojo.
- El punto de saturación es el **presupuesto máximo recomendado**: invertir más sube conversiones pero baja el ROI.
- **Limitaciones:** las curvas son extrapolaciones del modelo hasta 1,5× el gasto máximo observado
  por anuncio; asume que el CPM y el CTR de cada segmento se mantienen.

**Cómo lo usa el dashboard (para tu compañero):**
```python
from recomendador import RecomendadorInversion
reco = RecomendadorInversion.desde_archivos("modelo_regresion.pkl", "datos_limpios.csv", valor_conversion=100)
tabla, resumen = reco.recomendar(presupuesto, solo_rentable=True)
```

## 11. A/B testing

*Responsable: Deivid Julián Alvarado Morán*
*Responsable: Deivid*

**Pregunta:** ¿las diferencias de rendimiento entre campañas, géneros y rangos de edad son **reales** o
pueden deberse al azar?

El dataset no viene de un experimento aleatorizado: cada anuncio ya tenía asignada su campaña y su
público. Por eso esta sección es un **A/B testing observacional**: se compara cada par de grupos
(A vs B) con pruebas de hipótesis, pero una diferencia significativa indica *asociación*, no
necesariamente causa (las campañas también difieren en gasto, intereses, etc.).

| Métrica | Unidad | Prueba global (≥ 3 grupos) | Prueba por pares (A vs B) | Tamaño de efecto |
|---|---|---|---|---|
| Tasa de conversión (anuncio con ≥ 1 conversión aprobada) | anuncio | Chi-cuadrado | Prueba z de dos proporciones | h de Cohen |
| CTR | anuncio | ANOVA o Kruskal-Wallis | t de Welch o Mann-Whitney | d de Cohen / r biserial |
| CPA | anuncio con ≥ 1 conversión | ANOVA o Kruskal-Wallis | t de Welch o Mann-Whitney | d de Cohen / r biserial |

**Regla de decisión para CTR y CPA:** se aplica Shapiro-Wilk a cada grupo. Si **todos** los grupos
comparados son normales (p > α) se usa la prueba paramétrica (t de Welch; Levene decide si se asumen
varianzas iguales). Si alguno no es normal, se usa la no paramétrica (Mann-Whitney / Kruskal-Wallis).

**Comparaciones múltiples:** al comparar muchos pares sube la probabilidad de un falso positivo. Se
aplica **Bonferroni** dentro de cada familia (métrica × factor): α ajustado = 0,05 / número de pares.
También se reporta la corrección global sobre todas las comparaciones, más estricta.

**Potencia y tamaño de muestra:** para cada comparación se calcula la potencia alcanzada con el α de
Bonferroni, el tamaño de muestra que haría falta para detectar esa diferencia con 80 % de potencia
y el efecto mínimo detectable con la muestra actual.

In [ ]:
from itertools import combinations
from scipy import stats
from statsmodels.stats.proportion import (proportions_ztest, proportion_effectsize,
                                          proportion_confint)
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.multitest import multipletests

ALPHA = 0.05
POTENCIA_OBJETIVO = 0.80
FACTORES_AB = {"campaign_id": "Campaña", "age": "Edad", "gender": "Género"}
MIN_N_GRUPO = 3          # Shapiro necesita al menos 3 observaciones

ab = df.copy()
ab["convierte"] = (ab["approved_conversion"] >= 1).astype(int)
ab_cpa = ab[ab["approved_conversion"] > 0].copy()      # el CPA sólo existe si hubo conversiones

poder_prop, poder_t = NormalIndPower(), TTestIndPower()

for factor, nombre in FACTORES_AB.items():
    print(f"{nombre}: {ab[factor].value_counts().sort_index().to_dict()}")
print(f"\nAnuncios totales: {len(ab)} | con al menos una conversión (para CPA): {len(ab_cpa)}")

### 11.1 Descriptivo por grupo

Antes de las pruebas, el valor de cada métrica por grupo. La tasa de conversión se acompaña de su
intervalo de confianza de Wilson al 95 %. También se muestra la tasa de conversión **por clic**
(conversiones aprobadas / clics, el KPI de la sección 2) como referencia.

In [ ]:
def descriptivo(factor):
    g = ab.groupby(factor)
    d = pd.DataFrame({
        "anuncios": g.size(),
        "anuncios_que_convierten": g["convierte"].sum(),
        "tasa_conversion": g["convierte"].mean(),
        "conv_por_clic": g["approved_conversion"].sum() / g["clicks"].sum().replace(0, np.nan),
        "ctr_mediana": g["ctr"].median(),
        "ctr_media": g["ctr"].mean(),
        "cpa_mediana": ab_cpa.groupby(factor)["cpa"].median(),
        "cpa_media": ab_cpa.groupby(factor)["cpa"].mean(),
        "n_cpa": ab_cpa.groupby(factor).size(),
    })
    ic = proportion_confint(d["anuncios_que_convierten"], d["anuncios"], alpha=ALPHA, method="wilson")
    d.insert(3, "ic95_inf", ic[0]); d.insert(4, "ic95_sup", ic[1])
    d["n_cpa"] = d["n_cpa"].fillna(0).astype(int)
    return d

descriptivos = {f: descriptivo(f) for f in FACTORES_AB}
for factor, nombre in FACTORES_AB.items():
    print(f"\n=== {nombre} ===")
    display(descriptivos[factor].round(4))

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(16, 13))
for j, (factor, nombre) in enumerate(FACTORES_AB.items()):
    d = descriptivos[factor]
    err = np.vstack([d["tasa_conversion"] - d["ic95_inf"], d["ic95_sup"] - d["tasa_conversion"]])
    ax[0, j].bar(d.index.astype(str), d["tasa_conversion"], yerr=err, capsize=5, color="#5B6CFF")
    ax[0, j].set_title(f"Tasa de conversión por {nombre.lower()} (IC 95 %)")
    ax[0, j].set_ylabel("% de anuncios que convierten")
    ax[0, j].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

    orden = sorted(ab[factor].unique())
    sns.boxplot(data=ab, x=factor, y="ctr", order=orden, ax=ax[1, j], color="#9AA5FF", showfliers=False)
    ax[1, j].set_title(f"CTR por {nombre.lower()} (sin atípicos)"); ax[1, j].set_xlabel("")

    sns.boxplot(data=ab_cpa, x=factor, y="cpa", order=orden, ax=ax[2, j], color="#FFB199", showfliers=False)
    ax[2, j].set_title(f"CPA por {nombre.lower()} (sin atípicos)"); ax[2, j].set_xlabel(nombre)
plt.tight_layout(); plt.show()

### 11.2 Tasa de conversión: chi-cuadrado y prueba z

**Hipótesis por par:** H₀: p_A = p_B (los dos grupos convierten en la misma proporción de anuncios)
vs H₁: p_A ≠ p_B.

- **Chi-cuadrado** (tabla grupo × convierte) dice si hay diferencia en *algún* grupo del factor.
  Supuesto: frecuencias esperadas ≥ 5 en al menos el 80 % de las celdas (se reporta).
  El tamaño de efecto global es la **V de Cramér**.
- **Prueba z de dos proporciones** compara cada par A vs B. El tamaño de efecto es la **h de Cohen**
  (0,2 pequeño · 0,5 mediano · 0,8 grande).

Se usa el anuncio como unidad porque los anuncios son independientes entre sí; la tasa por clic
agregada no lo permite (varios clics del mismo anuncio no son observaciones independientes).

In [ ]:
def potencia_y_muestra(efecto, n_a, n_b, alfa, motor):
    # Potencia alcanzada, n por grupo necesario (80 %) y efecto mínimo detectable con la muestra actual
    efecto = abs(efecto)
    if not np.isfinite(efecto) or efecto < 1e-6 or min(n_a, n_b) < 2:
        return np.nan, np.nan, np.nan
    escalar = lambda v: float(np.asarray(v, dtype=float).ravel()[0])
    def seguro(f):
        try:
            v = escalar(f())
            return v if np.isfinite(v) and v > 0 else np.nan
        except Exception:
            return np.nan
    potencia = seguro(lambda: motor.power(effect_size=efecto, nobs1=n_a, alpha=alfa, ratio=n_b / n_a))
    if not np.isfinite(potencia):          # efectos muy grandes: la t no centrada no converge
        potencia = seguro(lambda: NormalIndPower().power(effect_size=efecto, nobs1=n_a, alpha=alfa,
                                                         ratio=n_b / n_a))
    n_req = seguro(lambda: motor.solve_power(effect_size=efecto, power=POTENCIA_OBJETIVO, alpha=alfa, ratio=1))
    mde = seguro(lambda: motor.solve_power(nobs1=n_a, power=POTENCIA_OBJETIVO, alpha=alfa, ratio=n_b / n_a))
    return potencia, (np.ceil(n_req) if np.isfinite(n_req) else np.nan), mde


def h_a_puntos(p_base, h):
    # Convierte una h de Cohen en diferencia absoluta de proporciones desde p_base
    p_nueva = np.sin(np.clip(np.arcsin(np.sqrt(p_base)) + h / 2, 0, np.pi / 2)) ** 2
    return p_nueva - p_base


omnibus, pares = [], []
for factor, nombre in FACTORES_AB.items():
    tabla = pd.crosstab(ab[factor], ab["convierte"]).reindex(columns=[0, 1], fill_value=0)
    chi2, p, gl, esperadas = stats.chi2_contingency(tabla)
    v_cramer = np.sqrt(chi2 / (tabla.values.sum() * (min(tabla.shape) - 1)))
    omnibus.append({"tipo": "global", "metrica": "Tasa de conversión", "factor": nombre,
                    "grupo_a": "todos", "grupo_b": "", "prueba": "Chi-cuadrado",
                    "estadistico": chi2, "gl": gl, "p_valor": p,
                    "tamano_efecto": v_cramer, "tipo_efecto": "V de Cramér",
                    "n_a": int(tabla.values.sum()),
                    "nota": f"{(esperadas < 5).mean():.0%} de celdas con esperada < 5"})

    resumen = ab.groupby(factor)["convierte"].agg(["sum", "count"])
    lista_pares = list(combinations(resumen.index, 2))
    alfa_bonf = ALPHA / len(lista_pares)
    for a, b in lista_pares:
        x = resumen.loc[[a, b], "sum"].to_numpy()
        n = resumen.loc[[a, b], "count"].to_numpy()
        pa, pb = x / n
        if x.sum() in (0, n.sum()):        # todos convierten o ninguno: la prueba z no aplica
            z, p = np.nan, 1.0
        else:
            z, p = proportions_ztest(x, n)
        h = proportion_effectsize(pa, pb)
        potencia, n_req, mde_h = potencia_y_muestra(h, n[0], n[1], alfa_bonf, poder_prop)
        pares.append({"tipo": "pareada", "metrica": "Tasa de conversión", "factor": nombre,
                      "grupo_a": str(a), "grupo_b": str(b), "n_a": int(n[0]), "n_b": int(n[1]),
                      "valor_a": pa, "valor_b": pb, "diferencia": pa - pb,
                      "prueba": "z de dos proporciones", "estadistico": z, "p_valor": p,
                      "alfa_bonferroni": alfa_bonf, "tamano_efecto": h, "tipo_efecto": "h de Cohen",
                      "potencia": potencia, "n_requerido_por_grupo": n_req,
                      "efecto_minimo_detectable": mde_h,
                      "mde_puntos_porcentuales": h_a_puntos(min(pa, pb), mde_h) * 100
                                                 if np.isfinite(mde_h) else np.nan})

display(pd.DataFrame(omnibus)[["factor", "estadistico", "gl", "p_valor", "tamano_efecto", "nota"]].round(4))

### 11.3 CTR y CPA: Shapiro → t de Welch o Mann-Whitney

**Hipótesis por par:** H₀: las dos distribuciones son iguales (mismo CTR / CPA) vs H₁: difieren.

1. **Shapiro-Wilk** en cada grupo (H₀: los datos son normales).
2. Si todos los grupos del par son normales → **Levene** (varianzas) y **t de Welch** (o t de Student
   si las varianzas son iguales). Efecto: **d de Cohen**.
3. Si alguno no es normal → **Mann-Whitney U**. Efecto: **r biserial de rangos** (0,1 pequeño ·
   0,3 mediano · 0,5 grande).
4. Para factores con 3 o más grupos se añade la prueba global: **ANOVA** si todos son normales,
   **Kruskal-Wallis** si no.

Para la potencia de Mann-Whitney se convierte la r biserial en una **d equivalente**
(si P(A > B) = (r + 1)/2, entonces d = √2 · Φ⁻¹((r + 1)/2)) y se usa la fórmula de la prueba t. Así la
potencia refleja el efecto en rangos que realmente mide la prueba y no la media, que los atípicos
distorsionan. El efecto mínimo detectable de CTR y CPA queda en unidades de d.

> Con más de ~50 datos, Shapiro detecta desviaciones mínimas de la normalidad; en métricas de
> marketing (muy asimétricas) es esperable que rechace casi siempre, lo que justifica las pruebas no
> paramétricas.

In [ ]:
def es_normal(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < MIN_N_GRUPO or np.ptp(x) == 0:
        return False, np.nan
    _, p = stats.shapiro(x if len(x) <= 5000 else np.random.default_rng(SEED).choice(x, 5000, replace=False))
    return p > ALPHA, p


def d_cohen(a, b):
    sa, sb = np.var(a, ddof=1), np.var(b, ddof=1)
    sp = np.sqrt(((len(a) - 1) * sa + (len(b) - 1) * sb) / (len(a) + len(b) - 2))
    return (np.mean(a) - np.mean(b)) / sp if sp > 0 else 0.0


normalidad = []
for metrica, datos in [("CTR", ab), ("CPA", ab_cpa)]:
    col = metrica.lower()
    for factor, nombre in FACTORES_AB.items():
        grupos = {str(k): v[col].dropna().to_numpy() for k, v in datos.groupby(factor)}
        grupos = {k: v for k, v in grupos.items() if len(v) >= MIN_N_GRUPO}
        shapiro = {k: es_normal(v) for k, v in grupos.items()}
        for k, (normal, p_sh) in shapiro.items():
            normalidad.append({"metrica": metrica, "factor": nombre, "grupo": k, "n": len(grupos[k]),
                               "shapiro_p": p_sh, "normal": normal})

        # Prueba global para factores con 3 o más grupos
        if len(grupos) >= 3:
            todos_normales = all(n for n, _ in shapiro.values())
            if todos_normales:
                est, p = stats.f_oneway(*grupos.values()); prueba = "ANOVA"
            else:
                est, p = stats.kruskal(*grupos.values()); prueba = "Kruskal-Wallis"
            n_tot = sum(len(v) for v in grupos.values())
            eps2 = est / (n_tot - 1) if prueba == "Kruskal-Wallis" else np.nan
            omnibus.append({"tipo": "global", "metrica": metrica, "factor": nombre,
                            "grupo_a": "todos", "grupo_b": "", "prueba": prueba,
                            "estadistico": est, "gl": len(grupos) - 1, "p_valor": p,
                            "tamano_efecto": eps2, "tipo_efecto": "épsilon² (Kruskal)" if prueba == "Kruskal-Wallis" else "",
                            "n_a": n_tot, "nota": "todos normales" if todos_normales else "algún grupo no normal"})

        lista_pares = list(combinations(sorted(grupos), 2))
        if not lista_pares:
            continue
        alfa_bonf = ALPHA / len(lista_pares)
        for a, b in lista_pares:
            xa, xb = grupos[a], grupos[b]
            normales = shapiro[a][0] and shapiro[b][0]
            d = d_cohen(xa, xb)
            if normales:
                _, p_lev = stats.levene(xa, xb)
                igual_var = p_lev > ALPHA
                est, p = stats.ttest_ind(xa, xb, equal_var=igual_var)
                prueba = "t de Student" if igual_var else "t de Welch"
                efecto, tipo_ef, nota = d, "d de Cohen", f"Levene p={p_lev:.3g}"
            else:
                est, p = stats.mannwhitneyu(xa, xb, alternative="two-sided")
                prueba = "Mann-Whitney U"
                efecto = 2 * est / (len(xa) * len(xb)) - 1          # r biserial de rangos
                tipo_ef, nota = "r biserial de rangos", "potencia con d equivalente a la r biserial"
            # Para Mann-Whitney se usa la d equivalente a la r biserial: P(A > B) = Φ(d / √2)
            d_pot = d if normales else np.sqrt(2) * stats.norm.ppf(np.clip((efecto + 1) / 2, 1e-6, 1 - 1e-6))
            potencia, n_req, mde_d = potencia_y_muestra(d_pot, len(xa), len(xb), alfa_bonf, poder_t)
            pares.append({"tipo": "pareada", "metrica": metrica, "factor": nombre,
                          "grupo_a": a, "grupo_b": b, "n_a": len(xa), "n_b": len(xb),
                          "valor_a": np.median(xa), "valor_b": np.median(xb),
                          "diferencia": np.median(xa) - np.median(xb),
                          "media_a": np.mean(xa), "media_b": np.mean(xb),
                          "shapiro_p_a": shapiro[a][1], "shapiro_p_b": shapiro[b][1],
                          "prueba": prueba, "estadistico": est, "p_valor": p,
                          "alfa_bonferroni": alfa_bonf, "tamano_efecto": efecto, "tipo_efecto": tipo_ef,
                          "d_cohen": d, "d_potencia": d_pot, "potencia": potencia, "n_requerido_por_grupo": n_req,
                          "efecto_minimo_detectable": mde_d, "nota": nota})

normalidad = pd.DataFrame(normalidad)
print("Grupos normales según Shapiro:", int(normalidad["normal"].sum()), "de", len(normalidad))
display(normalidad.round(4))

### 11.4 Corrección de Bonferroni y tabla final

Para cada familia (métrica × factor) con *m* comparaciones, una diferencia es significativa si
p × m < 0,05 (equivale a p < 0,05 / m). La columna `significativo_global` usa la corrección sobre
**todas** las comparaciones pareadas a la vez, para ver qué hallazgos resisten el criterio más estricto.

In [ ]:
res_pares = pd.DataFrame(pares)
res_pares["p_bonferroni"] = np.nan
for _, idx in res_pares.groupby(["metrica", "factor"]).groups.items():
    rechaza, p_aj, _, _ = multipletests(res_pares.loc[idx, "p_valor"].fillna(1), alpha=ALPHA, method="bonferroni")
    res_pares.loc[idx, "p_bonferroni"] = p_aj
    res_pares.loc[idx, "significativo"] = rechaza
_, p_global, _, _ = multipletests(res_pares["p_valor"].fillna(1), alpha=ALPHA, method="bonferroni")
res_pares["p_bonferroni_global"] = p_global
res_pares["significativo_global"] = p_global < ALPHA
res_pares["significativo"] = res_pares["significativo"].astype(bool)

res_global = pd.DataFrame(omnibus)
_, p_aj, _, _ = multipletests(res_global["p_valor"].fillna(1), alpha=ALPHA, method="bonferroni")
res_global["p_bonferroni"] = p_aj
res_global["significativo"] = p_aj < ALPHA


def conclusion(f):
    if f["tipo"] == "global":
        return (f"Hay diferencias en {f['metrica']} entre grupos de {f['factor'].lower()}"
                if f["significativo"] else f"No hay evidencia de diferencias en {f['metrica']} por {f['factor'].lower()}")
    if not f["significativo"]:
        return "Sin diferencia significativa"
    mejor_bajo = f["metrica"] == "CPA"                       # en CPA, menor es mejor
    gana = f["grupo_a"] if (f["valor_a"] < f["valor_b"]) == mejor_bajo else f["grupo_b"]
    return f"Gana {gana}"


ab_resultados = pd.concat([res_global, res_pares], ignore_index=True)
ab_resultados["conclusion"] = ab_resultados.apply(conclusion, axis=1)
columnas = ["tipo", "metrica", "factor", "grupo_a", "grupo_b", "n_a", "n_b", "valor_a", "valor_b",
            "diferencia", "prueba", "estadistico", "gl", "p_valor", "alfa_bonferroni", "p_bonferroni",
            "significativo", "p_bonferroni_global", "significativo_global", "tamano_efecto",
            "tipo_efecto", "d_cohen", "d_potencia", "potencia", "n_requerido_por_grupo", "efecto_minimo_detectable",
            "mde_puntos_porcentuales", "media_a", "media_b", "shapiro_p_a", "shapiro_p_b",
            "conclusion", "nota"]
ab_resultados = ab_resultados.reindex(columns=columnas)

vista = ["metrica", "factor", "grupo_a", "grupo_b", "valor_a", "valor_b", "prueba",
         "p_valor", "p_bonferroni", "significativo", "tamano_efecto", "potencia", "conclusion"]
display(ab_resultados[ab_resultados["tipo"] == "pareada"][vista].round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
for i, metrica in enumerate(["Tasa de conversión", "CTR", "CPA"]):
    d = res_pares[res_pares["metrica"] == metrica].copy()
    d["par"] = d["factor"] + ": " + d["grupo_a"] + " vs " + d["grupo_b"]
    d = d.sort_values("p_bonferroni", ascending=False)
    colores = np.where(d["significativo"], "#2BB673", "#B0B7C3")
    ax[i].barh(d["par"], -np.log10(d["p_bonferroni"].clip(lower=1e-300)), color=colores)
    ax[i].axvline(-np.log10(ALPHA), color="red", ls="--", label="α = 0,05")
    ax[i].set_title(f"{metrica}: −log10(p Bonferroni)"); ax[i].set_xlabel("más a la derecha = más evidencia")
ax[0].legend(); plt.tight_layout(); plt.show()

### 11.5 Potencia y tamaño de muestra

- **Potencia alcanzada:** probabilidad de detectar la diferencia observada con el n actual y el α de
  Bonferroni. Por debajo de 0,8 la prueba está *subdimensionada*: un "no significativo" puede ser
  falta de datos y no ausencia de efecto.
- **n requerido por grupo:** anuncios por grupo necesarios para detectar esa misma diferencia con
  80 % de potencia.
- **Efecto mínimo detectable (MDE):** la diferencia más pequeña que la muestra actual puede detectar
  con 80 % de potencia (en tasa de conversión se traduce a puntos porcentuales).

Al final, una tabla de **planificación**: cuántos anuncios por grupo necesitaría una prueba A/B
nueva para detectar distintas mejoras sobre la tasa de conversión actual.

In [ ]:
potencia_tabla = res_pares[["metrica", "factor", "grupo_a", "grupo_b", "n_a", "n_b", "tamano_efecto",
                            "potencia", "n_requerido_por_grupo", "efecto_minimo_detectable",
                            "mde_puntos_porcentuales", "significativo"]].copy()
potencia_tabla["muestra_suficiente"] = potencia_tabla["potencia"] >= POTENCIA_OBJETIVO
display(potencia_tabla.round(3))

baja = potencia_tabla[(~potencia_tabla["significativo"]) & (potencia_tabla["potencia"] < POTENCIA_OBJETIVO)]
print(f"Comparaciones no significativas con potencia < {POTENCIA_OBJETIVO:.0%}: {len(baja)} "
      f"de {(~potencia_tabla['significativo']).sum()} (no se puede concluir que no haya diferencia).")

# Planificación de una prueba A/B nueva (dos variantes, una comparación)
p_base = ab["convierte"].mean()
plan = []
for subida_pp in [2, 5, 10, 15]:
    p_nueva = min(p_base + subida_pp / 100, 0.999)
    h = proportion_effectsize(p_nueva, p_base)
    for m in [1, 3, 6]:        # 1 comparación (A/B), 3 (tres campañas), 6 (cuatro edades)
        n = poder_prop.solve_power(effect_size=h, power=POTENCIA_OBJETIVO, alpha=ALPHA / m, ratio=1)
        plan.append({"tasa_base": round(p_base, 4), "mejora_pp": subida_pp, "tasa_objetivo": round(p_nueva, 4),
                     "comparaciones": m, "alfa_bonferroni": round(ALPHA / m, 4),
                     "anuncios_por_grupo": int(np.ceil(n))})
plan = pd.DataFrame(plan).pivot(index=["tasa_base", "mejora_pp", "tasa_objetivo"],
                                columns="comparaciones", values="anuncios_por_grupo")
plan.columns = [f"n por grupo ({c} {'comparaciones' if c > 1 else 'comparación'})" for c in plan.columns]
plan

### 11.6 Exportación

`salidas/ab_resultados.csv` tiene una fila por prueba:
- `tipo = global`: chi-cuadrado / ANOVA / Kruskal-Wallis por factor.
- `tipo = pareada`: cada comparación A vs B con su p-valor, p de Bonferroni, tamaño de efecto,
  potencia, n requerido y una `conclusion` en texto para mostrar en la pestaña 4 del dashboard.

`valor_a` / `valor_b` son la tasa de conversión (proporción) o la **mediana** de CTR / CPA; las medias
están en `media_a` / `media_b`.

In [ ]:
os.makedirs("salidas", exist_ok=True)
ab_resultados.to_csv("salidas/ab_resultados.csv", index=False)
normalidad.to_csv("salidas/ab_normalidad.csv", index=False)

significativas = ab_resultados[(ab_resultados["tipo"] == "pareada") & ab_resultados["significativo"]]
print(f"Pruebas guardadas: {len(ab_resultados)} | pares significativos tras Bonferroni: {len(significativas)}\n")
for _, f in significativas.iterrows():
    print(f"- {f['metrica']} | {f['factor']} {f['grupo_a']} vs {f['grupo_b']}: "
          f"{f['valor_a']:.4g} vs {f['valor_b']:.4g} (p Bonf = {f['p_bonferroni']:.2g}, "
          f"{f['tipo_efecto']} = {f['tamano_efecto']:.2f}) → {f['conclusion']}")

try:
    from google.colab import files
    files.download("salidas/ab_resultados.csv")
except ImportError:
    print("\nArchivo en salidas/ab_resultados.csv")

### Conclusiones A/B

*Datos: 761 anuncios válidos tras la limpieza (se eliminaron 382 filas corruptas); 353 tienen al menos
una conversión aprobada (base para el CPA). α = 0,05 con corrección de Bonferroni por familia.*

1. **Tasa de conversión (anuncios con ≥ 1 conversión aprobada):**
   - **Campaña:** hay diferencias (χ² p < 0,001; V de Cramér = 0,25). La campaña **1178** convierte en el
     **64,6 %** de sus anuncios frente al 44,4 % de la 916 (p Bonf = 0,018) y al 37,1 % de la 936
     (p Bonf < 0,001, h = 0,56). Entre 916 y 936 no hay diferencia significativa.
   - **Género:** los anuncios dirigidos a **hombres** convierten más: 52,5 % vs 35,6 % en mujeres
     (p Bonf < 0,001, h = 0,34).
   - **Edad:** sin diferencias significativas (χ² p = 0,21). La caída de 45-49 (37,4 % vs ~48 % en el resto)
     no alcanza significancia y esas comparaciones tienen potencia < 30 %.
2. **CTR:** Shapiro rechazó la normalidad en los 18 grupos (p < 0,001), por eso se usaron Mann-Whitney y
   Kruskal-Wallis. El CTR **aumenta con la edad**: 45-49 y 40-44 superan claramente a 30-34
   (r = −0,44 y −0,40; p Bonf < 0,001). Las **mujeres** tienen mayor CTR que los hombres (r = 0,25;
   p Bonf < 0,001). La campaña 936 supera a la 1178 (p Bonf = 0,009), aunque el efecto es pequeño (r = −0,13).
3. **CPA:** la campaña **1178 es mucho más cara por conversión** (mediana 34,2 vs ~1,5 en 916 y 936;
   r ≈ 0,85-0,90, p Bonf < 0,001). Por edad, **30-34** tiene el CPA más bajo (mediana 3,3) frente a
   35-39 (15,1) y 40-44 (12,1). Por género, las **mujeres** convierten mucho más barato (mediana 1,7 vs 11,9;
   p Bonf < 0,001).
4. **Potencia:** las 15 comparaciones no significativas tienen potencia < 80 %, así que no prueban que los
   grupos sean iguales. Ejemplo: para confirmar la diferencia de tasa de conversión entre 30-34 y 45-49
   harían falta ~495 anuncios por grupo (hoy hay 327 y 115).
5. **Lectura de negocio:** la 1178 consigue que más anuncios conviertan, pero a un costo por conversión
   ~20 veces mayor; 916 y 936 son más eficientes. Los hombres convierten en más anuncios, pero las mujeres
   hacen más clic y convierten más barato. Se recomienda priorizar presupuesto en 916/936, en el público
   30-34 y en mujeres, y validar con una prueba A/B controlada (≈ 400 anuncios por variante para detectar
   +10 puntos en la tasa de conversión).
6. **Limitación:** los datos son observacionales (sin asignación aleatoria) y las campañas difieren en
   gasto y públicos; las diferencias son asociaciones, no efectos causales.

## 12. Atribución y comparación de canales

*Responsable: Giovanny Andrés Zambrano*

**Pregunta de Negocio:** ¿Cómo debemos atribuir el mérito de las conversiones y distribuir el presupuesto cuando los canales (Campaña e Intereses) interactúan en el ecosistema?

### 12.1 Justificación Teórica y Definición de Canales
En la analítica digital convencional, los modelos de atribución multi-toque (*Multi-Touch Attribution* o MTA) requieren un identificador único de usuario (`cookie_id`, `user_id`) y una marca de tiempo (`timestamp`) que registre los múltiples puntos de contacto antes de una conversión (por ejemplo: clic en búsqueda orgánica -> clic en banner de Facebook -> conversión directa).

**Justificación Técnica:** El dataset de *Facebook Ads Campaign* carece de identificadores de usuario individuales y de secuencias cronológicas multi-toque. Cada registro representa el desempeño agregado de un anuncio aislado. En consecuencia, definimos la competencia estratégica a nivel de **Canal sintético: Interacción de Campaña × Interés Principal (`campaign_id` × `interest1`)**. Tratamos estas combinaciones como canales de adquisición independientes que compiten por capturar el presupuesto y generar conversiones.

### Modelos de Atribución Implementados:
1. **Último Contacto (Last Touch):** Asigna el 100 % del mérito de la conversión al anuncio (segmento) donde se completó la conversión aprobada.
2. **Atribución Lineal:** Asume una distribución uniforme del mérito. Dividimos de manera ponderada la conversión considerando la interacción de los componentes del canal.
3. **Valor de Shapley (Teoría de Juegos):** Modela las campañas e intereses como 'jugadores' dentro de una coalición cooperativa. Calcula el aporte marginal de cada canal a la tasa general de conversión aprobada de la cartera, garantizando una distribución matemáticamente justa y óptima del valor.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from itertools import combinations

# Recuperación y reconstrucción robusta de datos limpios
try:
    df_attr = df.copy()
    print("Cargando datos desde la variable 'df' activa en memoria.")
except NameError:
    if os.path.exists("salidas/datos_limpios.csv"):
        df_attr = pd.read_csv("salidas/datos_limpios.csv")
        print("Cargando datos desde 'salidas/datos_limpios.csv'.")
    else:
        print("No se detectó 'df' ni el archivo exportado. Reconstruyendo el dataframe automáticamente...")
        # Buscar y cargar el CSV crudo del dataset original
        try:
            ruta_csv = localizar_csv()
        except NameError:
            # Definición local de localización en caso de desconexión completa
            import glob
            locales = [p for p in glob.glob("*.csv") + glob.glob("data/*.csv")
                       if not p.startswith(("datos_limpios", "kpis_", "predicciones_", "ab_", "atribucion_"))]
            if locales:
                ruta_csv = locales[0]
            else:
                try:
                    import kagglehub
                    carpeta = kagglehub.dataset_download("madislemsalu/facebook-ad-campaign")
                    ruta_csv = glob.glob(os.path.join(carpeta, "**", "*.csv"), recursive=True)[0]
                except Exception:
                    raise FileNotFoundError("Por favor sube el archivo de Facebook Ads al entorno de Colab.")

        df_temp = pd.read_csv(ruta_csv)
        df_temp.columns = (df_temp.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_"))
        df_temp = df_temp.rename(columns={"xyz_campaign_id": "campaign_id", "interest": "interest1"})

        # Limpieza básica equivalente a la Parte 1
        NUMERICAS_AUX = ["impressions", "clicks", "spent", "total_conversion", "approved_conversion"]
        for c in NUMERICAS_AUX:
            df_temp[c] = pd.to_numeric(df_temp[c], errors="coerce")

        mascara = df_temp[["campaign_id", "age", "gender"] + NUMERICAS_AUX].notna().all(axis=1)
        mascara &= df_temp["gender"].astype(str).str.upper().isin(["M", "F"])
        mascara &= df_temp["age"].astype(str).str.match(r"^\d{2}-\d{2}$ ") | df_temp["age"].astype(str).str.match(r"^\d{2}-\d{2}$")
        df_attr = df_temp[mascara].copy()
        df_attr = df_attr[(df_attr["impressions"] > 0) & (df_attr["spent"] >= 0) & (df_attr["clicks"] >= 0)]
        if "ad_id" in df_attr.columns:
            df_attr = df_attr.drop_duplicates(subset="ad_id")

        df_attr["campaign_id"] = df_attr["campaign_id"].astype(int).astype(str)
        df_attr["gender"] = df_attr["gender"].str.upper()
        df_attr["interest1"] = pd.to_numeric(df_attr["interest1"], errors="coerce").fillna(-1).astype(int).astype(str)
        print("¡Dataframe de respaldo recreado exitosamente!")

# Definimos el Canal
df_attr["canal"] = df_attr["campaign_id"].astype(str) + " - Int_" + df_attr["interest1"].astype(str)

print(f"Canales únicos detectados: {df_attr['canal'].nunique()}")
df_attr[["canal", "approved_conversion", "spent", "clicks", "impressions"]].head()

### 12.2 Desarrollo de los Modelos de Atribución (Last Touch, Linear & Shapley)

Implementaremos los tres modelos directamente sobre las conversiones aprobadas reales de cada canal.

In [ ]:
# 1. Atribución de Último Contacto (Last Touch)
last_touch = df_attr.groupby("canal")["approved_conversion"].sum().reset_index()
last_touch.rename(columns={"approved_conversion": "atribucion_last_touch"}, inplace=True)

# 2. Atribución Lineal: el mérito se reparte por igual entre los puntos de contacto del canal
#    (campaña e interés). Ambos pertenecen al mismo canal, así que este recibe 0,5 + 0,5 = 100 %.
df_attr["atrib_lineal_campana"] = df_attr["approved_conversion"] * 0.5
df_attr["atrib_lineal_interes"] = df_attr["approved_conversion"] * 0.5
linear_touch = (df_attr.assign(atribucion_lineal=df_attr["atrib_lineal_campana"] + df_attr["atrib_lineal_interes"])
                .groupby("canal")["atribucion_lineal"].sum().reset_index())

# 3. Atribución de Shapley (Teoría de Juegos) adaptada de forma robusta
def calcular_shapley_canales(dataframe):
    canales = dataframe["canal"].unique()
    shapley_values = {}
    total_conv = dataframe["approved_conversion"].sum()

    for c in canales:
        conv_con_canal = dataframe[dataframe["canal"] == c]["approved_conversion"].sum()
        conv_sin_canal = total_conv - conv_con_canal

        v_canal_solo = conv_con_canal
        v_gran_coalicion = total_conv
        v_vacio = 0

        val_shapley = 0.5 * (v_canal_solo - v_vacio) + 0.5 * (v_gran_coalicion - conv_sin_canal)
        shapley_values[c] = max(0.0, val_shapley)

    suma_shapley = sum(shapley_values.values())
    if suma_shapley > 0:
        for k in shapley_values:
            shapley_values[k] = (shapley_values[k] / suma_shapley) * total_conv

    return pd.DataFrame(list(shapley_values.items()), columns=["canal", "atribucion_shapley"])

shapley_df = calcular_shapley_canales(df_attr)

# Integración de Modelos de Atribución
atribucion_completa = last_touch.merge(linear_touch, on="canal").merge(shapley_df, on="canal")
print("¡Modelos de atribución calculados con éxito!")
atribucion_completa.round(3).head(10)

### 12.2.1 ¿Por qué los tres modelos dan el mismo resultado?

Es el resultado esperado y es metodológicamente correcto. Los modelos de atribución reparten el mérito de una
conversión entre **varios puntos de contacto del recorrido de un usuario**. Este dataset no tiene identificador de
usuario ni secuencia de contactos: cada anuncio pertenece a un solo canal (campaña × interés) y su conversión no la
comparte con ningún otro canal. Por eso último contacto, lineal y Shapley convergen al valor directo registrado
por cada anuncio.

El valor de esta sección no está en el reparto del mérito, sino en **cruzar las conversiones atribuidas con el
gasto de cada canal** (CPA y ROI). Eso permite separar canales de alto volumen pero ineficientes (por ejemplo
`1178 - Int_28`) de canales de bajo costo y alto retorno (por ejemplo `936 - Int_29`).

### 12.3 Comparación de Métricas de Eficiencia por Canal (CTR, CPC, CPA, ROI)

Cruzaremos los modelos de atribución con el rendimiento histórico financiero de cada canal sintético.

In [ ]:
# Métricas de costo y rendimiento por canal
kpis_canal = df_attr.groupby("canal").agg(
    impressions=("impressions", "sum"),
    clicks=("clicks", "sum"),
    spent=("spent", "sum"),
    approved_conversion=("approved_conversion", "sum")
).reset_index()

# Unir con la atribución
comparativo = kpis_canal.merge(atribucion_completa, on="canal")

# Calcular KPIs de eficiencia tradicionales
comparativo["ctr"] = comparativo["clicks"] / comparativo["impressions"]
comparativo["cpc"] = comparativo["spent"] / comparativo["clicks"]
# Calculamos CPA bajo los tres modelos de atribución diferentes
comparativo["cpa_last_touch"] = comparativo["spent"] / comparativo["atribucion_last_touch"].replace(0, np.nan)
comparativo["cpa_lineal"] = comparativo["spent"] / comparativo["atribucion_lineal"].replace(0, np.nan)
comparativo["cpa_shapley"] = comparativo["spent"] / comparativo["atribucion_shapley"].replace(0, np.nan)

# Recuperar robustamente el valor supuesto de conversión
VALOR_CONVERSION = 100.0
try:
    if 'metricas' in globals():
        VALOR_CONVERSION = float(metricas["valor_conversion_supuesto"])
    elif os.path.exists("salidas/metricas_modelos.json"):
        import json
        with open("salidas/metricas_modelos.json", "r", encoding="utf-8") as f:
            meta = json.load(f)
            VALOR_CONVERSION = float(meta.get("valor_conversion_supuesto", 100.0))
except Exception:
    pass

# Calculamos ROI bajo el valor de conversión configurado
comparativo["roi_last_touch"] = (comparativo["atribucion_last_touch"] * VALOR_CONVERSION - comparativo["spent"]) / comparativo["spent"].replace(0, np.nan)
comparativo["roi_lineal"] = (comparativo["atribucion_lineal"] * VALOR_CONVERSION - comparativo["spent"]) / comparativo["spent"].replace(0, np.nan)
comparativo["roi_shapley"] = (comparativo["atribucion_shapley"] * VALOR_CONVERSION - comparativo["spent"]) / comparativo["spent"].replace(0, np.nan)

# Filtrar y ordenar por canales más representativos (top 15 en conversiones reales)
top_canales = comparativo.sort_values(by="approved_conversion", ascending=False).head(15).copy()

display(top_canales[["canal", "spent", "approved_conversion", "atribucion_shapley",
                     "ctr", "cpc", "cpa_shapley", "roi_shapley"]].round(4))

### 12.4 Visualización de Atribución y Comparación de Canales
Generaremos gráficos para comparar visualmente los tres modelos de atribución y los ROIs resultantes.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(15, 12))

# Gráfico 1: Comparativa de Métodos de Atribución
top_grafico = top_canales.head(10)
x = np.arange(len(top_grafico["canal"]))
ancho = 0.25

ax[0].bar(x - ancho, top_grafico["atribucion_last_touch"], ancho, label="Last Touch", color="#5B6CFF")
ax[0].bar(x, top_grafico["atribucion_lineal"], ancho, label="Lineal", color="#B0B7C3")
ax[0].bar(x + ancho, top_grafico["atribucion_shapley"], ancho, label="Shapley (Teoría Juegos)", color="#2BB673")

ax[0].set_title("Comparación del Volumen de Conversiones Atribuidas por Modelo", fontsize=14, fontweight="bold")
ax[0].set_xticks(x)
ax[0].set_xticklabels(top_grafico["canal"], rotation=35, ha="right")
ax[0].set_ylabel("Conversiones Atribuidas")
ax[0].legend()

# Gráfico 2: Desempeño de ROI bajo el Modelo de Shapley
sns.barplot(data=top_grafico, x="canal", y="roi_shapley", palette="viridis", ax=ax[1])
ax[1].axhline(0, color="black", lw=1.2, ls="--")
ax[1].set_title("Retorno de Inversión (ROI Shapley) por Canal", fontsize=14, fontweight="bold")
ax[1].set_ylabel("ROI")
ax[1].set_xlabel("Canal (Campaña - Interés)")
ax[1].set_xticklabels(top_grafico["canal"], rotation=35, ha="right")

plt.tight_layout()
plt.show()

### 12.5 Exportación de Resultados

Guardamos los resultados de atribución en la ruta especificada `salidas/atribucion_resultados.csv` para alimentar el dashboard de la app web.

In [ ]:
import os
os.makedirs("salidas", exist_ok=True)
comparativo.to_csv("salidas/atribucion_resultados.csv", index=False)
print("Archivo 'salidas/atribucion_resultados.csv' exportado exitosamente con", len(comparativo), "registros.")

## 13. Exportación de resultados para la app web

| Archivo | Lo usa |
|---|---|
| `datos_limpios.csv` | A/B testing, atribución, dashboard |
| `kpis_campana.csv`, `kpis_edad.csv`, `kpis_genero.csv` | Dashboard, comparación de canales |
| `modelo_conversion.pkl` | Dashboard (probabilidad de conversión) |
| `modelo_regresion.pkl` | Recomendación de inversión |
| `predicciones_test.csv` | Dashboard (sección del modelo) |
| `metricas_modelos.json` | Informe y dashboard |
| `ab_resultados.csv`, `ab_normalidad.csv` | Dashboard (pestaña A/B testing) |
| `atribucion_resultados.csv` | Dashboard (pestaña atribución) |
| `recomendador.py` | Dashboard (recomendación de inversión) |
| `recomendacion_ejemplo.csv` | Dashboard e informe |

In [ ]:
os.makedirs("salidas", exist_ok=True)
df.to_csv("salidas/datos_limpios.csv", index=False)
kpis_campana.to_csv("salidas/kpis_campana.csv")
kpis_edad.to_csv("salidas/kpis_edad.csv")
kpis_genero.to_csv("salidas/kpis_genero.csv")
test.to_csv("salidas/predicciones_test.csv", index=False)
tabla_reco.to_csv("salidas/recomendacion_ejemplo.csv", index=False)
joblib.dump(mejor_modelo, "salidas/modelo_conversion.pkl")
joblib.dump(mejor_regresor, "salidas/modelo_regresion.pkl")

metricas = {
    "valor_conversion_supuesto": VALOR_CONVERSION,
    "features_categoricas": CATEGORICAS,
    "features_numericas": NUM_FEATURES,
    "clasificacion": comparacion.reset_index().to_dict(orient="records"),
    "modelo_seleccionado": mejor_nombre,
    "regresion": comparacion_reg[["MAE", "R2"]].astype(float).round(4).reset_index().to_dict(orient="records"),
    "regresor_seleccionado": mejor_reg_nombre,
    "importancia_permutacion": importancias,
    "roi_sin_modelo": round(float(base["roi"]), 4),
    "top_porcentaje_optimo": int(mejor_top),
    "umbral_optimo": float(umbral_optimo),
    "roi_con_modelo": round(float(fila["roi"]), 4),
    "presupuesto_ejemplo": float(PRESUPUESTO),
    "recomendacion_ejemplo": resumen_reco,
    "comparacion_estrategias": estrategias.round(4).to_dict(orient="index"),
}
with open("salidas/metricas_modelos.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2, default=str)

print(sorted(os.listdir("salidas")))

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("salidas_proyecto", "zip", "salidas")
    files.download("salidas_proyecto.zip")
except ImportError:
    print("Fuera de Colab: los archivos están en la carpeta 'salidas/'.")

## 14. Conclusiones finales

*Valores con el supuesto de 100 USD por conversión aprobada. Fuente: salidas de las secciones 1 a 12.*

1. **Calidad de datos:** de 1.143 filas se eliminaron 382 corruptas (valores desplazados entre columnas); quedaron
   **761 anuncios válidos**, con 19.620 USD de gasto y 585 conversiones aprobadas (ROI global 1,98).
2. **Modelos de clasificación:** **Random Forest** fue el mejor, con ROC-AUC de **0,629** en prueba frente a 0,601 de
   AdaBoost. Es un poder predictivo moderado: con 761 anuncios, las variables disponibles explican sólo en parte si
   un anuncio convierte. Las variables más influyentes fueron los intereses, el CPC, las impresiones y la campaña.
3. **ROI del modelo:** invertir sólo en los anuncios con mayor probabilidad **no mejoró el ROI** (2,25 invirtiendo en
   todos los anuncios de prueba). El clasificador sirve para orientar decisiones, no para recortar la cartera.
4. **Recomendación de inversión:** con un presupuesto de **4.900 USD**, el reparto del recomendador logra un ROI
   esperado de **6,15** (350 conversiones), frente a 5,30 del reparto proporcional al histórico y 3,98 del reparto
   igual. Pone el 80 % en la campaña 1178 (hombres de 30 a 44 años) y el 19 % en la 936 (sobre todo mujeres de
   35 a 49 años).
5. **A/B testing:** la campaña 1178 convierte en más anuncios (64,6 % vs 37,1 % de la 936), pero su CPA mediano es
   unas 20 veces mayor. Los hombres convierten en más anuncios; las mujeres hacen más clic y convierten más barato.
   El rango 30-34 tiene el CPA más bajo.
6. **Atribución y canales:** sin recorridos de usuario, los tres modelos coinciden. Por eficiencia, la campaña 1178
   se lleva el 84,5 % del gasto y sólo el 64,6 % de las conversiones; canales como `936 - Int_29` (ROI 6,16) son mucho
   más rentables que `1178 - Int_28` (CPA 86,30 USD).
7. **App web:** el dashboard en Streamlit muestra los KPIs con filtros, la predicción de conversión, la recomendación
   de inversión y los resultados de A/B testing y atribución.
8. **Limitaciones:** datos observacionales (asociaciones, no causalidad), ROI dependiente del valor supuesto por
   conversión y modelos entrenados con pocos anuncios. Antes de mover el presupuesto real conviene validar con una
   prueba A/B controlada.